# Adrianic v4.3 — Binding + Graph/Latent-Factor + Comparative Field-Gate

This notebook integrates the two uploaded upgrade families into the v4.2 binding test.

The probe-chain composed target sentence remains omitted. Primary success is measured by true-target discrimination against matched wrong targets.

In [ ]:
from __future__ import annotations
from pathlib import Path
from collections import Counter, defaultdict
import argparse, copy, json, math, random, re, urllib.request
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

# -----------------------------------------------------------------------------
# Adrianic v4.3 — Binding-specific semantic test with two imported upgrades:
#   (1) pairwise + latent common-cause evidence control (v3.21 family)
#   (2) comparative replacement + honest address/template separation (field gate)
#
# Scientific boundary: this is a tiny recurrent LM + explicit memory controller.
# It tests functional relational binding under interference, not human understanding.
# -----------------------------------------------------------------------------

if Path('/content').exists(): ROOT = Path('/content')
elif Path('/mnt/data').exists(): ROOT = Path('/mnt/data')
else: ROOT = Path.cwd()
OUT = ROOT / 'adrianic_v43_binding_gate_results'
OUT.mkdir(parents=True, exist_ok=True)

TOKEN_RE = re.compile(r"[a-zA-Z]+|[0-9]+|[^\w\s]")
STOP = set('the a an is are in of to and this that with for near under before after through beyond fact remember'.split())

def seed_all(seed:int):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)

def tokenize(s:str): return TOKEN_RE.findall(s.lower())

def content_tokens(s:str):
    return [w for w in tokenize(s) if w.isalpha() and w not in STOP]

FACTS=[
('two plus two equals','four'),('three plus five equals','eight'),('seven minus three equals','four'),('six times seven equals','fortytwo'),
('the square root of nine is','three'),('a triangle has','three'),('a square has','four'),('the additive identity is','zero'),
('the si unit of force is the','newton'),('the si unit of energy is the','joule'),('the si unit of power is the','watt'),('mass times acceleration gives','force'),
('energy associated with motion is','kinetic'),('the force attracting masses is','gravity'),('the chemical symbol for hydrogen is','h'),
('the chemical symbol for oxygen is','o'),('water contains hydrogen and','oxygen'),('an atom with six protons is','carbon'),
('an atom with eight protons is','oxygen'),('the center of an atom is the','nucleus'),('earth orbits the','sun'),('the moon orbits the','earth'),
('the largest planet in the solar system is','jupiter'),('plants use light in','photosynthesis')]
REPHRASE=[
('in mechanics mass times acceleration produces','force'),('force is measured using the unit called','newton'),
('energy uses the si unit called','joule'),('water contains hydrogen plus','oxygen'),('six protons identifies','carbon'),
('eight protons identifies','oxygen'),('earth travels around the','sun'),('the moon travels around the','earth')]

SYL1=['nar','vel','tor','lum','kas','mer','sol','dav','pir','zan','fel','rin','bos','kel','vim','jor','tav','qul','ser','pan','dru','hel','cor','nis']
SYL2=['a','en','or','is','um','et','on','ar','il','os','ex','un']

def unique_words(n,seed):
    rng=random.Random(seed); pool=[a+b for a in SYL1 for b in SYL2]; rng.shuffle(pool); return pool[:n]

def make_chains(seed=0,n_demo=20,n_probe=10):
    names=unique_words((n_demo+n_probe)*3,seed+818); demo=[]; probe=[]; rows=[]; k=0
    for i in range(n_demo+n_probe):
        x,y,z=names[k:k+3]; k+=3
        premises=[f'{x} contains {y}.',f'{y} maps to {z}.']
        if i<n_demo:
            # Preserve v4.2 curriculum exactly: sentences are independent training items.
            demo += (premises+[f'{x} ultimately maps to {z}.',f'{y} is contained by {x}.'])*3
            rows.append(('demo',x,y,z))
        else:
            # Probe target and inverse sentences are deliberately absent.
            probe += premises*4
            rows.append(('probe',x,y,z))
    return demo,probe,pd.DataFrame(rows,columns=['kind','x','y','z'])

def make_training_corpus(seed=0,contextual_demo=False):
    rng=random.Random(seed); s=[]
    for p,a in FACTS: s += [f'{p} {a}.',f'fact: {p} {a}.',f'remember {p} {a}.']
    demo,probe,chains=make_chains(seed); s += demo+probe
    if contextual_demo:
        # Schema-ingestion patch: demonstration entities additionally appear in a contiguous
        # premise->premise->composition context. Probe entities still NEVER receive the
        # composed target or a combined two-premise paragraph during training.
        for _,r in chains[chains.kind=='demo'].iterrows():
            para=f'{r.x} contains {r.y}. {r.y} maps to {r.z}. {r.x} ultimately maps to {r.z}.'
            s += [para,para]
    for i in range(80): s.append(f'noise{i} drift{i} hollow{i} amber{i} sevenfold{i}.')
    rng.shuffle(s)
    return s,chains

def proxy_stream(seed=0,n=180):
    rng=random.Random(seed+500)
    names=['arden','cassian','lyra','orion','mara','selene','tavian','viola']
    verbs=['whispers','wanders','argues','laughs','waits','returns','dreams','speaks']
    objs=['beside the gate','through the hall','near the river','under the tower','after the feast','beyond the garden','before the crowd','under the moon']
    return [f'{rng.choice(names)} {rng.choice(verbs)} {rng.choice(objs)}.' for _ in range(n)]

def download_natural():
    urls={'shakespeare':'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt',
          'bible':'https://www.gutenberg.org/cache/epub/10900/pg10900.txt'}
    out={}
    for name,url in urls.items():
        p=ROOT/f'{name}_corpus.txt'
        if not p.exists():
            print('downloading',name); urllib.request.urlretrieve(url,p)
        text=p.read_text(errors='ignore')
        out[name]=[q.strip().replace('\n',' ') for q in re.split(r'(?<=[.!?])\s+|\n{2,}',text) if 4<=len(tokenize(q))<=70]
    return out

def natural_stream(seed=0,corpus='interleaved',per_source=700):
    d=download_natural(); rng=random.Random(seed+900)
    A=rng.sample(d['shakespeare'],min(per_source,len(d['shakespeare'])))
    B=rng.sample(d['bible'],min(per_source,len(d['bible'])))
    if corpus=='shakespeare': return A
    if corpus=='bible': return B
    out=[]; n=min(len(A),len(B))
    for a,b in zip(A[:n],B[:n]): out += ([a,b] if rng.random()<.5 else [b,a])
    return out

class Vocab:
    def __init__(self,texts,max_vocab=10000):
        c=Counter()
        for t in texts:c.update(tokenize(t))
        self.itos=['<pad>','<unk>']+[w for w,_ in c.most_common(max_vocab-2)]
        self.stoi={w:i for i,w in enumerate(self.itos)}
    def encode(self,s):return [self.stoi.get(w,1) for w in tokenize(s)]

class TinySSM(nn.Module):
    def __init__(self,V,emb=36,hid=56):
        super().__init__();self.emb=nn.Embedding(V,emb);self.rnn=nn.RNN(emb,hid,nonlinearity='tanh',batch_first=True);self.out=nn.Linear(hid,V)
    def forward(self,x,h=None):z,h=self.rnn(self.emb(x),h);return self.out(z),h

def dev(model):return next(model.parameters()).device

def train_sentence(model,opt,ids):
    if len(ids)<2:return np.nan
    d=dev(model);x=torch.tensor(ids[:-1],dtype=torch.long,device=d)[None,:];y=torch.tensor(ids[1:],dtype=torch.long,device=d)[None,:]
    opt.zero_grad(set_to_none=True);logits,_=model(x);loss=nn.functional.cross_entropy(logits.reshape(-1,logits.size(-1)),y.reshape(-1));loss.backward();torch.nn.utils.clip_grad_norm_(model.parameters(),1.0);opt.step();return float(loss.item())

def seq_loss(model,ids):
    if len(ids)<2:return 0.
    d=dev(model);x=torch.tensor(ids[:-1],dtype=torch.long,device=d)[None,:];y=torch.tensor(ids[1:],dtype=torch.long,device=d)[None,:]
    with torch.no_grad():logits,_=model(x)
    return float(nn.functional.cross_entropy(logits.reshape(-1,logits.size(-1)),y.reshape(-1)).item())

def logits_for_prompt(model,vocab,prompt):
    ids=vocab.encode(prompt);d=dev(model);x=torch.tensor(ids,dtype=torch.long,device=d)[None,:]
    with torch.no_grad():logits,_=model(x)
    return logits[0,-1]

def token_rank(model,vocab,prompt,answer):
    v=logits_for_prompt(model,vocab,prompt);aid=vocab.stoi.get(answer,1);return int((v>v[aid]).sum().item())+1

def top_tokens(model,vocab,prompt,k=5):
    v=logits_for_prompt(model,vocab,prompt);return [vocab.itos[i] for i in torch.topk(v,k=min(k,len(vocab.itos))).indices.tolist()]

def candidate_rank(model,vocab,prompt,true_answer,candidates):
    v=logits_for_prompt(model,vocab,prompt)
    scores=[]
    for a in candidates:
        scores.append((a,float(v[vocab.stoi.get(a,1)].item())))
    scores.sort(key=lambda x:x[1],reverse=True)
    order=[a for a,_ in scores]
    return order.index(true_answer)+1,dict(scores)

# -----------------------------------------------------------------------------
# Candidate evidence and v3.21-style dependency/factor inference
# -----------------------------------------------------------------------------

def unique_candidate_table(train,vocab,early_model,final_model):
    pos=defaultdict(list)
    for i,s in enumerate(train):pos[s].append(i)
    docs=list(pos.keys());N=len(train)
    df=Counter()
    for s in docs:df.update(set(content_tokens(s)))
    rows=[]
    for idx,s in enumerate(docs):
        toks=content_tokens(s);pp=pos[s]
        recurrence=math.log1p(len(pp))
        token_persistence=float(np.mean([math.log1p(df[t]) for t in toks])) if toks else 0.
        cross_context=float(np.mean([df[t]>=3 for t in toks])) if toks else 0.
        temporal_span=(max(pp)-min(pp))/max(1,N-1) if len(pp)>1 else 0.
        early_pred=-seq_loss(early_model,vocab.encode(s))
        final_pred=-seq_loss(final_model,vocab.encode(s))
        rows.append({'uid':idx,'text':s,'first_pos':min(pp),'last_pos':max(pp),'count':len(pp),
                     'recurrence':recurrence,'token_persistence':token_persistence,'cross_context':cross_context,
                     'temporal_span':temporal_span,'early_predictability':early_pred,'future_predictability':final_pred})
    return pd.DataFrame(rows)

SOURCE_COLS=['recurrence','token_persistence','cross_context','temporal_span','early_predictability']

def zcols(X):
    X=np.asarray(X,float);mu=X.mean(0);sd=X.std(0);active=sd>1e-8;Z=np.zeros_like(X)
    Z[:,active]=(X[:,active]-mu[active])/(sd[active]+1e-9)
    return Z,mu,sd,active

def connected_components_from_corr(C,threshold=.72):
    n=C.shape[0];adj={i:set() for i in range(n)}
    for i in range(n):
        for j in range(i+1,n):
            if C[i,j]>threshold:adj[i].add(j);adj[j].add(i)
    comps=[];seen=set()
    for i in range(n):
        if i in seen:continue
        st=[i];c=set()
        while st:
            x=st.pop()
            if x in seen:continue
            seen.add(x);c.add(x);st.extend(adj[x]-seen)
        if len(c)>=2:comps.append(c)
    return comps

def learn_multifactor(cov,maxf=3):
    n=cov.shape[0];vals,vecs=np.linalg.eigh(cov);o=np.argsort(vals)[::-1];vals=vals[o];vecs=vecs[:,o]
    base=np.median(vals[min(maxf,n):])+1e-9 if n>maxf else 1e-9
    groups=[];loads=[]
    for k in range(min(maxf,n)):
        strength=max(0.,float((vals[k]-base)/(vals[k]+1e-9)))
        v=np.abs(vecs[:,k]);mx=v.max()+1e-12;g={i for i,l in enumerate(v) if l/mx>.58}
        if len(g)>=2 and strength>.22 and not any(len(g&h)/len(g|h)>.75 for h in groups):
            groups.append(g);loads.append((v,strength))
    return groups,loads,vals

def cap_weights(raw,g):
    g=[i for i in g if raw[i]>0]
    if len(g)<=1:return raw
    total=sum(raw[i] for i in g);cap=max(raw[i] for i in g);f=cap/(total+1e-12)
    for i in g:raw[i]*=f
    return raw

def evidence_model(cand,mode='factor'):
    X=cand[SOURCE_COLS].to_numpy(float);Z,mu,sd,active=zcols(X)
    y=cand['future_predictability'].to_numpy(float);yz=(y-y.mean())/(y.std()+1e-9)
    residual=Z-yz[:,None]
    bias=residual.mean(0)
    rho=np.mean(np.exp(-np.abs(residual)/.8),axis=0)
    cov=np.cov((residual-residual.mean(0)).T)+np.eye(len(SOURCE_COLS))*1e-6
    rsd=np.sqrt(np.maximum(np.diag(cov),1e-9));corr=np.clip(cov/(rsd[:,None]*rsd[None,:]+1e-12),-1,1)
    comps=connected_components_from_corr(corr,.72)
    groups,loads,eigs=learn_multifactor(cov,3)
    raw={i:float(rho[i]**3) if active[i] else 0. for i in range(len(SOURCE_COLS))}
    if mode in ('graph','factor'):
        for g in comps:raw=cap_weights(raw,g)
    if mode=='factor':
        for g,(load,strength) in zip(groups,loads):
            raw=cap_weights(raw,g);mx=max(load[i] for i in g)+1e-12
            for i in g:raw[i]/=(1+1.15*strength*load[i]/mx)
    w=np.array([raw[i] for i in range(len(SOURCE_COLS))]);w/=w.sum()+1e-12
    deb=Z-bias[None,:];U=deb@w
    disagreement=np.sum(w[None,:]*(deb-U[:,None])**2,axis=1)
    # v2-style common-mode risk: don't 'correct' value; lower confidence when many channels miss together.
    substantial=np.abs(residual)>.8
    same_sign=np.maximum((residual>.8).sum(1),(residual<-.8).sum(1))
    h=float(np.mean(same_sign>=max(2,int(math.ceil(len(SOURCE_COLS)/2)))))
    conf=np.exp(-disagreement)*(1-min(.8,h))
    U01=1/(1+np.exp(-np.clip(U,-12,12)))
    out=cand.copy();out['utility_z']=U;out['utility01']=U01;out['evidence_confidence']=conf;out['future_yz']=yz
    meta={'source_cols':SOURCE_COLS,'rho':rho.tolist(),'bias':bias.tolist(),'weights':w.tolist(),'corr':corr.tolist(),
          'graph_components':[[SOURCE_COLS[i] for i in sorted(g)] for g in comps],
          'factor_groups':[[SOURCE_COLS[i] for i in sorted(g)] for g in groups],
          'factor_strengths':[float(x[1]) for x in loads],'factor_loadings':[x[0].tolist() for x in loads],
          'common_cause_risk':h,'eigenvalues':eigs.tolist()}
    return out,meta

# -----------------------------------------------------------------------------
# Parity-protected raw templates + comparative replacement bank
# Address != template: address is transparent IDF token signature; template is raw ids.
# -----------------------------------------------------------------------------

class Slot:
    PRIME=1000003
    def __init__(self,text,ids,uid,U,S,Q,count,first_pos,created_step):
        self.text=text;self.a=list(ids);self.b=list(ids);self.uid=uid;self.U=float(U);self.S=float(S);self.Q=float(Q);self.count=int(count);self.first_pos=int(first_pos);self.created_step=int(created_step)
        self.p,self.q=self.chk(self.a);self.corrupted=False
    @classmethod
    def chk(cls,ids):
        q=np.asarray(ids,dtype=np.int64);return int(q.sum()%cls.PRIME),int(((np.arange(len(q))+1)*q).sum()%cls.PRIME)
    def corrupt(self,rng):
        if self.a:
            j=rng.randrange(len(self.a));self.a[j]=(self.a[j]+rng.randint(1,19))%100000;self.corrupted=True
    def retrieve(self,parity=True):
        if self.chk(self.a)==(self.p,self.q):return list(self.a),'primary'
        if parity and self.chk(self.b)==(self.p,self.q):return list(self.b),'repaired'
        return None,'lost'

class SemanticBank:
    def __init__(self,capacity,vocab,idf,comparative=True,mode='factor'):
        self.capacity=capacity;self.vocab=vocab;self.idf=idf;self.comparative=comparative;self.mode=mode;self.slots=[];self.writes=0;self.replacements=0;self.blocked=0;self.holds=0;self.margin=0.
    def retained_value(self,slot,step):
        age=max(0,step-slot.created_step)/max(1,step+1)
        return slot.U*math.sqrt(max(1e-9,math.log1p(slot.count)))/(1+.15*age)
    def candidate_value(self,row):
        return float(row.utility01*(.65+.35*row.stability)*(.75+.25*row.evidence_confidence))
    def admit(self,row,step,margin):
        if row.evidence_confidence <= 0 or row.utility01<.5:
            self.holds+=1;return False
        c=self.candidate_value(row)
        slot=Slot(row.text,self.vocab.encode(row.text),int(row.uid),float(row.utility01),float(row.stability),float(row.evidence_confidence),int(row['count']),int(row.first_pos),step)
        if len(self.slots)<self.capacity:
            self.slots.append(slot);self.writes+=1;return True
        vals=[self.retained_value(s,step) for s in self.slots];j=int(np.argmin(vals))
        if self.comparative:
            if c-vals[j] <= margin:self.blocked+=1;return False
        else:
            # Old/automatic full-bank behavior: qualifying candidate evicts weakest.
            if c<.5:self.blocked+=1;return False
        self.slots[j]=slot;self.writes+=1;self.replacements+=1;return True
    def corrupt(self,p,seed):
        rng=random.Random(seed)
        for s in self.slots:
            if rng.random()<p:s.corrupt(rng)
    def replay_ids(self,parity=True):
        out=[];repair=lost=0
        for s in self.slots:
            ids,status=s.retrieve(parity)
            if ids is not None:out.append(ids)
            repair+=status=='repaired';lost+=status=='lost'
        return out,repair,lost
    def query_score(self,q_tokens,slot):
        q=set(q_tokens);s=set(content_tokens(slot.text));ov=q&s
        if not ov:return 0.
        num=sum(self.idf.get(t,1.) for t in ov);den=sum(self.idf.get(t,1.) for t in q)+1e-12
        return float(num/den)
    def retrieve_templates(self,prompt,max_hops=2,parity=True):
        # Honest associative retrieval: never force a guess on tied/no-overlap addresses.
        q=content_tokens(prompt);chosen=[];seen=set();logs=[]
        for hop in range(max_hops):
            scores=[]
            for i,s in enumerate(self.slots):
                if i in seen:continue
                scores.append((self.query_score(q,s),i))
            if not scores:break
            scores.sort(reverse=True);best,bi=scores[0];second=scores[1][0] if len(scores)>1 else 0.
            # Dynamic margin: rare-token exact overlap should separate from the next address.
            margin=max(.025,.10*best)
            if best<=0 or best-second<=margin:
                logs.append({'hop':hop,'resolved':False,'best':best,'second':second,'margin':margin});break
            slot=self.slots[bi];ids,status=slot.retrieve(parity)
            if ids is None:
                logs.append({'hop':hop,'resolved':False,'best':best,'second':second,'margin':margin,'status':'lost'});break
            chosen.append(slot.text);seen.add(bi);q += content_tokens(slot.text)
            logs.append({'hop':hop,'resolved':True,'best':best,'second':second,'margin':margin,'status':status,'text':slot.text,'uid':slot.uid})
        return chosen,logs


def compute_idf(cand):
    docs=[set(content_tokens(s)) for s in cand.text];df=Counter();
    for d in docs:df.update(d)
    N=len(docs);return {t:(math.log((N+1)/(c+1))+1)**2 for t,c in df.items()}

def add_stability(cand):
    x=cand.copy();span=x.temporal_span.to_numpy(float);cnt=np.log1p(x['count'].to_numpy(float));cnt=cnt/(cnt.max()+1e-12);x['stability']=np.clip(.5*span+.5*cnt,0,1);return x

def gate_margin(cand):
    # Field report says the margin is calibration-dependent, not universal.
    # Here it scales to candidate-value dispersion using no semantic labels.
    C=cand.utility01.to_numpy(float)*(.65+.35*cand.stability.to_numpy(float))*(.75+.25*cand.evidence_confidence.to_numpy(float))
    return float(.25*np.std(C))

def build_bank(cand,meta,vocab,kind='factor_gate',capacity=96):
    idf=compute_idf(cand);bank=SemanticBank(capacity,vocab,idf,comparative=('auto' not in kind),mode=kind)
    margin=gate_margin(cand);bank.margin=margin
    # Process in observational order, not best-first.
    for step,row in cand.sort_values('first_pos').iterrows():bank.admit(row,step,margin)
    return bank

def old_v42_bank(train,vocab,final_model,capacity=96):
    # v4.2 selector, but bank is unique-template bounded for fair address retrieval.
    pos=defaultdict(list)
    for i,s in enumerate(train):pos[s].append(i)
    docs=list(pos.keys());counts=Counter();toks=[]
    for s in docs:
        q=content_tokens(s);toks.append(q);counts.update(set(q))
    rows=[]
    for i,(s,q) in enumerate(zip(docs,toks)):
        rows.append({'uid':i,'text':s,'first_pos':min(pos[s]),'count':len(pos[s]),
                     'recurrence':np.mean([math.log1p(counts[w]) for w in q]) if q else 0.,
                     'cross_context':np.mean([counts[w]>=3 for w in q]) if q else 0.,
                     'predictability':-seq_loss(final_model,vocab.encode(s))})
    d=pd.DataFrame(rows);X=d[['recurrence','cross_context','predictability']].to_numpy(float);Z,_,_,active=zcols(X)
    C=np.zeros((3,3));idx=np.where(active)[0]
    if len(idx)>1:
        Q=np.nan_to_num(np.abs(np.corrcoef(Z[:,idx],rowvar=False)));np.fill_diagonal(Q,0)
        for a,i in enumerate(idx):
            for b,j in enumerate(idx):C[i,j]=Q[a,b]
    rho=np.where(active,.7,0);a=rho**3;base=a/(a.sum()+1e-12);w=a/(1+1.5*(C@base));w/=w.sum()+1e-12;score=Z@w
    sets=[set(content_tokens(s)) for s in docs];selected=[];remain=set(range(len(d)))
    while remain and len(selected)<min(capacity,len(d)):
        best=None;bv=-1e99
        for i in remain:
            red=max((len(sets[i]&sets[j])/(len(sets[i]|sets[j])+1e-12) for j in selected),default=0.);v=float(score[i])-.85*red
            if v>bv:bv=v;best=i
        selected.append(best);remain.remove(best)
    # Reuse bank container, direct add selected templates.
    idf=compute_idf(pd.DataFrame({'text':docs}));bank=SemanticBank(capacity,vocab,idf,comparative=False,mode='v42_selector')
    for step,i in enumerate(selected):
        row=d.iloc[i];slot=Slot(row.text,vocab.encode(row.text),int(row.uid),.75,.75,.75,int(row['count']),int(row.first_pos),step);bank.slots.append(slot);bank.writes+=1
    return bank

def random_bank(train,vocab,capacity,seed):
    docs=list(dict.fromkeys(train));rng=random.Random(seed);picks=rng.sample(docs,min(capacity,len(docs)));tmp=pd.DataFrame({'text':docs});idf=compute_idf(tmp);bank=SemanticBank(capacity,vocab,idf,comparative=False,mode='random')
    pos={s:i for i,s in enumerate(docs)}
    for step,s in enumerate(picks):bank.slots.append(Slot(s,vocab.encode(s),pos[s],.5,.5,.5,1,pos[s],step));bank.writes+=1
    return bank

# -----------------------------------------------------------------------------
# Binding-specific probes. No chain ids/answers enter training or memory selection.
# -----------------------------------------------------------------------------

def augment_prompt(bank,prompt,parity=True,order='path'):
    if bank is None:return prompt,[],[]
    templates,logs=bank.retrieve_templates(prompt,2,parity)
    if order=='reverse':templates=list(reversed(templates))
    aug=(' '.join(templates)+' '+prompt).strip() if templates else prompt
    return aug,templates,logs

def probe_panel(model,vocab,chains,seed,stage,variant,bank=None,parity=True):
    rows=[]
    def add(kind,prompt,ans,item,mode='plain',candidates=None,wrong=None):
        aug=prompt;retr=[];logs=[]
        if mode in ('assist','assist_reverse') and bank is not None:
            aug,retr,logs=augment_prompt(bank,prompt,parity,'reverse' if mode=='assist_reverse' else 'path')
        elif mode=='second_only' and bank is not None:
            # Retrieve two hops, but keep only the second retrieved template. This is a recency/copy control.
            _,rr,ll=augment_prompt(bank,prompt,parity,'path');retr=rr[-1:] if rr else [];logs=ll;aug=(' '.join(retr)+' '+prompt).strip() if retr else prompt
        r=token_rank(model,vocab,aug,ans);cr=np.nan;margin=np.nan;wrong_score=np.nan;true_score=np.nan
        if candidates:
            cr,sc=candidate_rank(model,vocab,aug,ans,candidates);true_score=sc.get(ans,np.nan)
            if wrong is not None:wrong_score=sc.get(wrong,np.nan);margin=true_score-wrong_score
        rows.append({'seed':seed,'stage':stage,'variant':variant,'probe':kind,'mode':mode,'item':item,'prompt':prompt,'aug_prompt':aug,'answer':ans,
                     'rank':r,'candidate_rank':cr,'binding_margin':margin,'true_score':true_score,'wrong_score':wrong_score,
                     'top1':int(r==1),'top5':int(r<=5),'candidate_top1':int(cr==1) if not np.isnan(cr) else np.nan,
                     'top_tokens':'|'.join(top_tokens(model,vocab,aug,5)),'retrieved':' || '.join(retr),'retrieval_log':json.dumps(logs)})
    for i,(p,a) in enumerate(FACTS):add('exact_fact',p,a,f'fact{i}')
    for i,(p,a) in enumerate(REPHRASE):add('rephrase',p,a,f'rephrase{i}')
    q=chains[chains.kind=='probe'].reset_index(drop=True);zcands=q.z.tolist();xcands=q.x.tolist()
    for i,row in q.iterrows():
        wrong=q.iloc[(i+1)%len(q)]
        for mode in ['plain','assist','assist_reverse','second_only']:
            add('two_hop',f'{row.x} ultimately maps to',row.z,f'chain{i}',mode,zcands,wrong.z)
        add('two_hop_shuffled',f'{row.x} ultimately maps to',wrong.z,f'chain{i}','plain',zcands,row.z)
        for mode in ['plain','assist']:
            add('inverse',f'{row.y} is contained by',row.x,f'chain{i}',mode,xcands,wrong.x)
        add('premise_contains',f'{row.x} contains',row.y,f'chain{i}')
        add('premise_maps',f'{row.y} maps to',row.z,f'chain{i}')
    return pd.DataFrame(rows)

def summarize_probes(df):
    def safe_cmrr(x):
        a=np.asarray(x,float);a=a[np.isfinite(a)]
        return float(np.mean(1/a)) if len(a) else np.nan
    return df.groupby(['stage','variant','probe','mode']).agg(top1=('top1','mean'),top5=('top5','mean'),candidate_top1=('candidate_top1','mean'),
        mrr=('rank',lambda x:float(np.mean(1/np.asarray(x,float)))),candidate_mrr=('candidate_rank',safe_cmrr),
        binding_margin=('binding_margin','mean'),median_rank=('rank','median'),n=('rank','size')).reset_index()

VARIANTS=['baseline','random_bank','v42_selector','v43_graph_gate','v43_factor_auto','v43_factor_gate']

def run_one(seed=0,full=False,corpus='interleaved',device=None,capacity=96,contextual_demo=False):
    seed_all(seed);train,chains=make_training_corpus(seed,contextual_demo=contextual_demo);stream=natural_stream(seed,corpus,700) if full else proxy_stream(seed,180);vocab=Vocab(train+stream)
    if device is None:device='cuda' if full and torch.cuda.is_available() else 'cpu'

    # Shared initial learner: all variants branch from identical weights.
    base=TinySSM(len(vocab.itos)).to(device);opt=torch.optim.Adam(base.parameters(),lr=.006);order=list(range(len(train)));early_state=None
    for ep in range(3):
        random.Random(seed+ep).shuffle(order)
        for i in order:train_sentence(base,opt,vocab.encode(train[i]))
        if ep==0:early_state=copy.deepcopy(base.state_dict())
    final_state=copy.deepcopy(base.state_dict())
    early=TinySSM(len(vocab.itos)).to(device);early.load_state_dict(early_state)
    cand=unique_candidate_table(train,vocab,early,base);cand=add_stability(cand)
    cand_graph,meta_graph=evidence_model(cand,'graph');cand_factor,meta_factor=evidence_model(cand,'factor')

    # Banks are built without semantic/fact/chain labels.
    banks={
      'baseline':None,
      'random_bank':random_bank(train,vocab,capacity,seed+999),
      'v42_selector':old_v42_bank(train,vocab,base,capacity),
      'v43_graph_gate':build_bank(cand_graph,meta_graph,vocab,'graph_gate',capacity),
      'v43_factor_auto':build_bank(cand_factor,meta_factor,vocab,'factor_auto',capacity),
      'v43_factor_gate':build_bank(cand_factor,meta_factor,vocab,'factor_gate',capacity),
    }
    runs=[];probes=[];bank_audit=[]
    qprobe=chains[chains.kind=='probe'].reset_index(drop=True)
    premise_text=set()
    for _,r in qprobe.iterrows():premise_text|={f'{r.x} contains {r.y}.',f'{r.y} maps to {r.z}.'}

    for variant in VARIANTS:
        model=TinySSM(len(vocab.itos)).to(device);model.load_state_dict(final_state);opt=torch.optim.Adam(model.parameters(),lr=.006);bank=banks[variant]
        # Evaluate pre-interference plain behavior only once per variant.
        probes.append(probe_panel(model,vocab,chains,seed,'pre',variant,None,True))
        repair=lost=0;replay=[]
        if bank is not None:
            bank.corrupt(.18,seed+1234);replay,repair,lost=bank.replay_ids(True)
        total=0
        for ep in range(2 if full else 1):
            for s in stream:
                train_sentence(model,opt,vocab.encode(s));total+=1
                if bank is not None and replay and total%6==0:train_sentence(model,opt,replay[(total//6)%len(replay)])
        post=probe_panel(model,vocab,chains,seed,'post',variant,bank,True);probes.append(post)

        def mrr(kind,mode='plain'):
            q=post[(post.probe==kind)&(post['mode']==mode)];return float(np.mean(1/q['rank'].to_numpy(float))) if len(q) else np.nan
        def cmrr(kind,mode):
            q=post[(post.probe==kind)&(post['mode']==mode)];x=q.candidate_rank.to_numpy(float);return float(np.nanmean(1/x)) if len(q) else np.nan
        def ctop(kind,mode):
            q=post[(post.probe==kind)&(post['mode']==mode)];return float(np.nanmean(q.candidate_top1)) if len(q) else np.nan
        def bmargin(kind,mode):
            q=post[(post.probe==kind)&(post['mode']==mode)];return float(np.nanmean(q.binding_margin)) if len(q) else np.nan
        # direct true-vs-shuffled margin plus transparent address-path audit.
        plain=post[(post.probe=='two_hop')&(post['mode']=='plain')]
        assist=post[(post.probe=='two_hop')&(post['mode']=='assist')]
        rev=post[(post.probe=='two_hop')&(post['mode']=='assist_reverse')]
        second=post[(post.probe=='two_hop')&(post['mode']=='second_only')]
        target_hits=[];two_template=[]
        for _,ar in assist.iterrows():
            rt='' if pd.isna(ar.retrieved) else str(ar.retrieved)
            target_hits.append(str(ar.answer).lower() in set(content_tokens(rt)))
            two_template.append('||' in rt)
        if bank is not None:
            stored={s.text for s in bank.slots};premise_cov=len(stored & premise_text)/max(1,len(premise_text))
            both=0
            for _,r in qprobe.iterrows():both+=int(f'{r.x} contains {r.y}.' in stored and f'{r.y} maps to {r.z}.' in stored)
            both/=len(qprobe)
            ba={'seed':seed,'variant':variant,'capacity':capacity,'slots':len(bank.slots),'writes':bank.writes,'replacements':bank.replacements,'blocked':bank.blocked,'holds':bank.holds,'margin':bank.margin,'premise_template_coverage':premise_cov,'both_premises_stored':both,'repair':repair,'lost':lost}
            bank_audit.append(ba)
        runs.append({'seed':seed,'variant':variant,'full':full,'corpus':corpus,
          'exact_mrr':mrr('exact_fact'),'rephrase_mrr':mrr('rephrase'),'premise_contains_mrr':mrr('premise_contains'),'premise_maps_mrr':mrr('premise_maps'),
          'twohop_plain_mrr':mrr('two_hop','plain'),'twohop_assist_mrr':mrr('two_hop','assist'),'twohop_reverse_mrr':mrr('two_hop','assist_reverse'),'twohop_second_only_mrr':mrr('two_hop','second_only'),
          'twohop_plain_candidate_mrr':cmrr('two_hop','plain'),'twohop_assist_candidate_mrr':cmrr('two_hop','assist'),'twohop_reverse_candidate_mrr':cmrr('two_hop','assist_reverse'),'twohop_second_only_candidate_mrr':cmrr('two_hop','second_only'),
          'twohop_plain_candidate_top1':ctop('two_hop','plain'),'twohop_assist_candidate_top1':ctop('two_hop','assist'),'twohop_reverse_candidate_top1':ctop('two_hop','assist_reverse'),'twohop_second_only_candidate_top1':ctop('two_hop','second_only'),
          'binding_margin_plain':bmargin('two_hop','plain'),'binding_margin_assist':bmargin('two_hop','assist'),'binding_margin_reverse':bmargin('two_hop','assist_reverse'),'binding_margin_second_only':bmargin('two_hop','second_only'),
          'retrieval_target_hit_rate':float(np.mean(target_hits)) if target_hits else 0.,'retrieval_two_template_rate':float(np.mean(two_template)) if two_template else 0.,
          'inverse_plain_mrr':mrr('inverse','plain'),'inverse_assist_mrr':mrr('inverse','assist'),'parity_repaired':repair,'parity_lost':lost,
          'stream_loss':float(np.mean([seq_loss(model,vocab.encode(s)) for s in stream[-60:]]))})

    evidence_audit={'seed':seed,'graph':meta_graph,'factor':meta_factor}
    return pd.DataFrame(runs),pd.concat(probes,ignore_index=True),pd.DataFrame(bank_audit),cand_factor,evidence_audit

def main(argv=None):
    ap=argparse.ArgumentParser();ap.add_argument('--full',action='store_true');ap.add_argument('--corpus',choices=['shakespeare','bible','interleaved'],default='interleaved');ap.add_argument('--seeds',type=int,default=3);ap.add_argument('--capacity',type=int,default=96);ap.add_argument('--contextual-demo',action='store_true',help='Add contiguous demonstration-chain contexts; probe targets remain omitted.');args=ap.parse_args(argv)
    rr=[];pp=[];bb=[];cc=[];ee=[]
    for seed in range(args.seeds):
        print('seed',seed);r,p,b,c,e=run_one(seed,args.full,args.corpus,capacity=args.capacity,contextual_demo=args.contextual_demo);rr.append(r);pp.append(p);bb.append(b);c['seed']=seed;cc.append(c);ee.append(e)
        print(r[['variant','twohop_plain_candidate_mrr','twohop_assist_candidate_mrr','binding_margin_plain','binding_margin_assist']].to_string(index=False))
    runs=pd.concat(rr,ignore_index=True);probes=pd.concat(pp,ignore_index=True);banks=pd.concat(bb,ignore_index=True);cand=pd.concat(cc,ignore_index=True);tag=(('full_'+args.corpus) if args.full else 'proxy')+('_contextual' if args.contextual_demo else '_legacy')
    runs.to_csv(OUT/f'{tag}_{args.seeds}seed_runs.csv',index=False);probes.to_csv(OUT/f'{tag}_{args.seeds}seed_probe_details.csv',index=False);banks.to_csv(OUT/f'{tag}_{args.seeds}seed_bank_audit.csv',index=False);cand.to_csv(OUT/f'{tag}_{args.seeds}seed_candidate_evidence.csv',index=False);(OUT/f'{tag}_{args.seeds}seed_evidence_models.json').write_text(json.dumps(ee,indent=2));summarize_probes(probes).to_csv(OUT/f'{tag}_{args.seeds}seed_probe_summary.csv',index=False)
    med=runs.groupby('variant').median(numeric_only=True)
    primary='v43_factor_gate'
    def vals(v,col): return runs[runs.variant==v].sort_values('seed')[col].to_numpy(float)
    checks={
      'primary_plain_binding_margin_positive_median':bool(med.loc[primary,'binding_margin_plain']>0),
      'primary_assisted_binding_margin_positive_median':bool(med.loc[primary,'binding_margin_assist']>0),
      'primary_assisted_binding_margin_positive_seed_fraction':float(np.mean(vals(primary,'binding_margin_assist')>0)),
      'primary_assisted_candidate_mrr_above_plain_seed_fraction':float(np.mean(vals(primary,'twohop_assist_candidate_mrr')>vals(primary,'twohop_plain_candidate_mrr'))),
      'primary_assisted_beats_random_candidate_mrr_seed_fraction':float(np.mean(vals(primary,'twohop_assist_candidate_mrr')>vals('random_bank','twohop_assist_candidate_mrr'))),
      'factor_gate_beats_graph_gate_assisted_seed_fraction':float(np.mean(vals(primary,'twohop_assist_candidate_mrr')>vals('v43_graph_gate','twohop_assist_candidate_mrr'))),
      'comparative_gate_beats_factor_auto_assisted_seed_fraction':float(np.mean(vals(primary,'twohop_assist_candidate_mrr')>vals('v43_factor_auto','twohop_assist_candidate_mrr'))),
      'full_context_beats_second_only_seed_fraction':float(np.mean(vals(primary,'twohop_assist_candidate_mrr')>vals(primary,'twohop_second_only_candidate_mrr'))),
      'reverse_margin_positive_seed_fraction':float(np.mean(vals(primary,'binding_margin_reverse')>0)),
      'primary_retrieval_target_hit_rate_median':float(med.loc[primary,'retrieval_target_hit_rate']),
      'parity_repairs_faults':bool(med.loc[primary,'parity_repaired']>0),
    }
    (OUT/f'{tag}_{args.seeds}seed_checks.json').write_text(json.dumps(checks,indent=2))
    bankmed=banks.groupby('variant').median(numeric_only=True) if len(banks) else pd.DataFrame()
    report=f'''Adrianic v4.3 — Binding + Graph/Latent-Factor + Comparative Field-Gate Test\n\nMode: {tag}\nSeeds: {args.seeds}\nCapacity: {args.capacity}\nContextual demonstration ingestion: {args.contextual_demo}\n\nWhat changed from v4.2\n======================\n1. Binding is now scored directly among the ten candidate Z targets. The primary measures are candidate MRR, candidate top-1, and true-vs-matched-wrong logit margin.\n2. A v3.21-style evidence controller learns source reliability, residual correlation, pairwise dependency components, and up to three latent residual factors from a self-supervised future-predictability anchor. No semantic/fact/chain labels enter this layer.\n3. A field-gate-derived bounded bank uses comparative replacement. The replacement margin scales to candidate-value dispersion rather than reusing the field benchmark's 0.28, which its own report said was calibration-dependent.\n4. Address and template are separate: retrieval uses a transparent IDF-weighted lexical address; the stored template remains the parity-protected raw token sequence. Retrieval may return unresolved rather than force an ambiguous address.\n5. At probe time the bank can perform up to two associative retrieval hops. This is evaluated separately from the plain RNN response.\n\nControls\n========\n- baseline: no bank/replay\n- random_bank: matched capacity with random templates\n- v42_selector: previous evidence/diversity selector\n- v43_graph_gate: pairwise graph + comparative gate\n- v43_factor_auto: graph+latent factors + automatic replacement\n- v43_factor_gate: graph+latent factors + comparative replacement (primary)\n\nThe two-hop composed target sentence remains absent for every probe chain.\n\nMedian run metrics\n==================\n{med[['exact_mrr','rephrase_mrr','premise_contains_mrr','premise_maps_mrr','twohop_plain_candidate_mrr','twohop_assist_candidate_mrr','twohop_reverse_candidate_mrr','twohop_second_only_candidate_mrr','twohop_plain_candidate_top1','twohop_assist_candidate_top1','binding_margin_plain','binding_margin_assist','binding_margin_reverse','binding_margin_second_only','retrieval_target_hit_rate','retrieval_two_template_rate','inverse_plain_mrr','inverse_assist_mrr','stream_loss','parity_repaired','parity_lost']].to_string()}\n\nMedian bank audit\n=================\n{bankmed.to_string() if len(bankmed) else 'none'}\n\nChecks\n======\n{json.dumps(checks,indent=2)}\n\nInterpretation discipline\n=========================\nA positive assisted result can mean that a transparent external memory retrieved the correct premises and the recurrent learner used them. It is NOT equivalent to proving that the RNN internally discovered first-order logic.\n\nThe crucial anti-cheating control is second_only: if full two-premise context does not beat providing only the retrieved Y->Z sentence, then apparent two-hop success may be mostly recency/copy behavior rather than composition. Reverse-context and matched-wrong margins provide additional binding checks.\n\nThe v3.21 factor thresholds and the semantic address/replacement calibration remain scaffolds. They are logged for ablation rather than hidden.\n'''
    (OUT/f'{tag}_{args.seeds}seed_REPORT.txt').write_text(report);print(report)



## Primary full natural-language interference run

```python
main(["--full", "--corpus", "interleaved", "--seeds", "3", "--contextual-demo"])
```

In [ ]:
main(["--full", "--corpus", "interleaved", "--seeds", "3", "--contextual-demo"])

## Legacy-ingestion control

Run separately if you want to isolate the effect of contextual schema ingestion:

```python
main(["--full", "--corpus", "interleaved", "--seeds", "3"])
```

## Verified local proxy result

# Adrianic v4.3 — Binding + Graph/Latent-Factor + Comparative Field-Gate

## Local three-seed proxy verification

This is a mechanics / binding integration test, not a substitute for the full Shakespeare + KJV run.

### Integrated upgrades

- v3.21-style source reliability, pairwise residual dependency and multi-factor residual grouping.
- common-cause confidence reduction / unresolved behavior.
- bounded comparative replacement rather than automatic full-bank eviction.
- separate transparent address state from parity-protected raw template state.
- honest associative retrieval that may return unresolved.
- binding-specific true-vs-wrong scoring.

A small **contextual schema-ingestion patch** was also added: demonstration entities additionally appear in contiguous premise → premise → composition contexts. Probe entities still never receive the composed target sentence.

### Primary local result

Median two-hop candidate MRR:

- v43 factor-gate plain: **0.306786**
- v43 factor-gate assisted: **0.337778**
- random-bank assisted: **0.283730**
- graph-only gate assisted: **0.255000**
- latent-factor auto-replacement assisted: **0.291071**

The assisted true-vs-matched-wrong binding margin was positive in **all 3 seeds**.

Per-seed primary results:

```text
 seed  twohop_plain_candidate_mrr  twohop_assist_candidate_mrr  binding_margin_assist
    0                    0.306786                     0.359405               0.247358
    1                    0.198016                     0.337778               0.081637
    2                    0.323889                     0.243889               0.111812
```

The primary assisted candidate MRR beat the matched random bank in **all 3 seeds**.

### Memory addressing / binding behavior

The correct Z token appeared in retrieved memory context in **80% pooled** probe cases.
Median per-seed target-hit rate was **90%**.

Median both-premises-stored rate was **90%**.

This is evidence that the explicit memory controller is doing useful relation-specific address/binding work.

### Comparative replacement

Median latent-factor automatic replacement:
- writes: **115**
- replacements: **19**

Median latent-factor comparative gate:
- writes: **104**
- replacements: **8**

The gate therefore reduced memory churn substantially while beating the auto-replacement variant on assisted candidate MRR in **67%** of seeds.

### The critical residual

Full two-premise context beat the **second-premise-only** control in only **33% of seeds**.

So v4.3 does **not** yet establish an internal two-hop composition operator.

The defensible reading is:

> the upgraded controller is much better at storing and retrieving the relation-specific target context, and that improves true-target discrimination; however, much of the observed gain can still be explained by retrieval of the Y → Z premise itself.

That is meaningful **functional binding through explicit memory**, but it is not yet proof that the recurrent learner itself performs compositional logic.

### Robust checks

```json
{
  "assisted_binding_margin_positive_all_3_seeds": true,
  "assisted_candidate_mrr_beats_random_all_3_seeds": true,
  "assisted_mrr_above_plain_seed_fraction": 0.6666666666666666,
  "factor_gate_beats_graph_gate_seed_fraction": 0.6666666666666666,
  "comparative_gate_beats_factor_auto_seed_fraction": 0.6666666666666666,
  "full_context_beats_second_only_seed_fraction": 0.3333333333333333,
  "reverse_margin_positive_seed_fraction": 0.6666666666666666,
  "pooled_retrieval_target_hit_rate": 0.8,
  "median_retrieval_target_hit_rate": 0.9
}
```

### Full natural-corpus command

```python
main(["--full", "--corpus", "interleaved", "--seeds", "3", "--contextual-demo"])
```

Legacy-ingestion control:

```python
main(["--full", "--corpus", "interleaved", "--seeds", "3"])
```
